## Step 5: Secondary Scaffolding with Ragout + Cactus
**Input:** SPAdes assembly scaffolds; 7 reference Fusarium oxysporum genomes 
(Fo47, ZUM2407, V032g, ME23, LD-06, Fo5176, Race4)  
**Output:** Chromosome-scale scaffolds in `06-scaffolding/ragout_maf_output/`; 
whole-genome alignment in `06-scaffolding/cactus/alignment.maf`  
**Tools:** Mash v2.3, Cactus v2.9.9, Ragout v2.3  
**Workflow:** (1) Mash genome distances → UPGMA tree; 
(2) Cactus whole-genome alignment (HAL → MAF); 
(3) Ragout MAF-based scaffolding  
**Key finding:** Contigs reduced from 1,043 to 12; 
N50 improved from 738 kb to 3.52 Mb; largest contig 7.58 Mb  
**Reference:** Materials & Methods Section 4.3 — Nebli et al. (2025)

# Configuration and Setup

In [ ]:
export SN=3RR
export NCPUS=128

In [ ]:

alias ragout="apptainer run docker://ghcr.io/nexomis/ragout:build_1.0-ragout_2.3-hal_2.3-Sibelia_3.0.7 ragout"
alias mash="apptainer run docker://staphb/mash:2.3-CBIRDv2 mash"

In [ ]:
mkdir -p 06-scaffolding/references
mkdir -p 06-scaffolding/ragout_output

# Data Preparation

In [ ]:
wget -P 06-scaffolding/references https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/013/085/055/GCF_013085055.1_ASM1308505v1/GCF_013085055.1_ASM1308505v1_genomic.fna.gz
wget -P 06-scaffolding/references https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/048/165/035/GCA_048165035.1_ASM4816503v1/GCA_048165035.1_ASM4816503v1_genomic.fna.gz
wget -P 06-scaffolding/references https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/048/164/945/GCA_048164945.1_ASM4816494v1/GCA_048164945.1_ASM4816494v1_genomic.fna.gz
wget -P 06-scaffolding/references https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/030/719/095/GCA_030719095.1_ASM3071909v1/GCA_030719095.1_ASM3071909v1_genomic.fna.gz
---------------------------
wget -P 06-scaffolding/references https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/054/643/155/GCA_054643155.1_ASM5464315v1/GCA_054643155.1_ASM5464315v1_genomic.fna.gz #LD-06
wget -P 06-scaffolding/references https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/030/345/115/GCA_030345115.2_ASM3034511v2/GCA_030345115.2_ASM3034511v2_genomic.fna.gz # Fo5176
wget -P 06-scaffolding/references https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/027/920/445/GCA_027920445.1_ASM2792044v1/GCA_027920445.1_ASM2792044v1_genomic.fna.gz #race4



In [ ]:
# Decompress and rename reference files
gunzip 06-scaffolding/references/*.gz
mv 06-scaffolding/references/GCF_013085055.1_ASM1308505v1_genomic.fna 06-scaffolding/references/Fo47.fasta
mv 06-scaffolding/references/GCA_048165035.1_ASM4816503v1_genomic.fna 06-scaffolding/references/ZUM2407.fasta
mv 06-scaffolding/references/GCA_048164945.1_ASM4816494v1_genomic.fna 06-scaffolding/references/V032g.fasta
mv 06-scaffolding/references/GCA_030719095.1_ASM3071909v1_genomic.fna 06-scaffolding/references/ME23.fasta
-----------------------------
mv 06-scaffolding/references/GCA_054643155.1_ASM5464315v1_genomic.fna 06-scaffolding/references/LD-06.fasta
mv 06-scaffolding/references/GCA_030345115.2_ASM3034511v2_genomic.fna 06-scaffolding/references/fo5176.fasta
mv 06-scaffolding/references/GCA_027920445.1_ASM2792044v1_genomic.fna 06-scaffolding/references/race4.fasta

# Rename fasta headers for Fo47 to have cleaner names (e.g., chr_I, chr_II)
sed -i -E 's/>.*chromosome ([IVX]+),.*/>chr_\1/' 06-scaffolding/references/Fo47.fasta
cp 04-assembly/spades/option-A/scaffolds.fasta 06-scaffolding/references/${SN}-scaffolds.fasta

# Building a Phylogenetic Tree, fast from genome sequences

Step-by-step with Mash and Python

In [ ]:
mash sketch -k 32 -s 100000 -o 06-scaffolding/references/sketch.msh 06-scaffolding/references/*fasta
mash dist -s 100000 06-scaffolding/references/sketch.msh 06-scaffolding/references/*fasta > 06-scaffolding/mash_dist.tsv

In [ ]:
#!/usr/bin/env python
import pandas as pd
import numpy as np
from scipy.cluster.hierarchy import linkage, to_tree
from scipy.spatial.distance import squareform
import os

def get_newick(node, newick, parent_dist, leaf_names):
    """
    Convert a SciPy linkage matrix to a Newick format string.
    """
    if node.is_leaf():
        return "%s:%.4f%s" % (leaf_names[node.id], parent_dist - node.dist, newick)
    else:
        if len(newick) > 0:
            newick = ")%s" % newick
        else:
            newick = ");"
        dist = parent_dist - node.dist
        if dist < 0:
            dist = 0
        newick = get_newick(node.get_left(), ",%s" % get_newick(node.get_right(), "", node.dist, leaf_names), node.dist, leaf_names) + newick
        newick = "(%s:%.4f" % (newick, dist)
        return newick

# Read the distance matrix
dist_df = pd.read_csv('06-scaffolding/mash_dist.tsv', sep='\t', header=None, names=['ref1', 'ref2', 'dist', 'pvalue', 'kmers'])

# Pivot to create a square distance matrix
dist_pivot = dist_df.pivot(index='ref1', columns='ref2', values='dist').fillna(0)
labels = [os.path.basename(name).replace('.fasta', '') for name in dist_pivot.index]
dist_pivot.index = labels
dist_pivot.columns = labels

# Ensure the matrix is symmetric and convert to condensed format
dist_matrix = dist_pivot.values
dist_matrix = (dist_matrix + dist_matrix.T) / 2
np.fill_diagonal(dist_matrix, 0)
condensed_dist = squareform(dist_matrix)

# Perform UPGMA clustering
Z = linkage(condensed_dist, 'average')

# Convert to Newick format
tree = to_tree(Z)
newick_tree = get_newick(tree, "", tree.dist, labels)

# Save the tree to a file
with open('06-scaffolding/mash.nwk', 'w') as f:
    f.write(newick_tree)

print("Newick tree saved to 06-scaffolding/mash.nwk")
print(newick_tree)

# Phylogenetic scaffolding

## Prepare Ragout Recipe File

In [ ]:

# # Create the recipe file
# cat > 06-scaffolding/ragout_recipe.rcp << EOF
# .references = Fo47,ZUM2407,V032g,ME23
# .target = sample10_contigs
# .naming_ref = Fo47

# Fo47.fasta = /home/fbouzid/SN/06-scaffolding/references/Fo47.fasta
# ZUM2407.fasta = /home/fbouzid/SN/06-scaffolding/references/ZUM2407.fasta
# V032g.fasta = /home/fbouzid/SN/06-scaffolding/references/V032g.fasta
# ME23.fasta = /home/fbouzid/SN/06-scaffolding/references/ME23.fasta
# sample10_contigs.fasta = /home/fbouzid/SN/06-scaffolding/references/sample10_contigs.fasta

# EOF

## Ragout with Cactus/MAF (used)

In [ ]:
alias cactus="apptainer run docker://quay.io/comparative-genomics-toolkit/cactus:v2.9.9 cactus"
pip install toil

# Create directory for Cactus workflow
mkdir -p 06-scaffolding/cactus

## using nwk file from Mash

cat > 06-scaffolding/cactus/genomes.txt << EOF
(sample10-scaffolds,((LD-06,race4),(ME23,(ZUM2407,(V032g,(Fo47,fo5176))))));
sample10-scaffolds ${PWD}/06-scaffolding/references/sample10-scaffolds.fasta
Fo47 ${PWD}/06-scaffolding/references/Fo47.fasta
ZUM2407 ${PWD}/06-scaffolding/references/ZUM2407.fasta
V032g ${PWD}/06-scaffolding/references/V032g.fasta
ME23 ${PWD}/06-scaffolding/references/ME23.fasta
LD-06 ${PWD}/06-scaffolding/references/LD-06.fasta
fo5176 ${PWD}/06-scaffolding/references/fo5176.fasta
race4 ${PWD}/06-scaffolding/references/race4.fasta
EOF

In [ ]:
# Run Cactus to create HAL alignment
cactus 06-scaffolding/cactus/js \
  06-scaffolding/cactus/genomes.txt \
  06-scaffolding/cactus/alignment.hal \
  --maxCores 128


# Convert HAL to MAF Format

In [ ]:
# Set up cactus-hal2maf alias
alias cactus-hal2maf="apptainer run docker://quay.io/comparative-genomics-toolkit/cactus:v2.9.9 cactus-hal2maf"

# Convert HAL to MAF format
cactus-hal2maf 06-scaffolding/cactus/js-hal2maf \
  06-scaffolding/cactus/alignment.hal \
  06-scaffolding/cactus/alignment.maf \
  --refGenome Fo47 \
  --filterGapCausingDupes \
  --noAncestors \
  --dupeMode single \
  --chunkSize 500000

# Create Ragout Recipe for MAF Method

In [ ]:
# Create MAF-based recipe file
cat > 06-scaffolding/ragout_maf_recipe.rcp << EOF
.references = Fo47,ZUM2407,V032g,ME23,LD-06,fo5176,race4
.target = sample10-scaffolds
.naming_ref = Fo47
.maf = ${PWD}/06-scaffolding/cactus/alignment.maf

Fo47.fasta = /home/fbouzid/SN/06-scaffolding/references/Fo47.fasta
ZUM2407.fasta = /home/fbouzid/SN/06-scaffolding/references/ZUM2407.fasta
V032g.fasta = /home/fbouzid/SN/06-scaffolding/references/V032g.fasta
ME23.fasta = /home/fbouzid/SN/06-scaffolding/references/ME23.fasta
sample10-scaffolds.fasta = /home/fbouzid/SN/06-scaffolding/references/sample10-scaffolds.fasta
LD-06.fasta = /home/fbouzid/06-scaffolding/references/LD-06.fasta
fo5176.fasta = /home/fbouzid/06-scaffolding/references/fo5176.fasta
race4.fasta = /home/fbouzid/06-scaffolding/references/race4.fasta
EOF


In [ ]:
ragout 06-scaffolding/ragout_maf_recipe.rcp \
  -o 06-scaffolding/ragout_maf_output \
  -s maf \
  -t $NCPUS \
  --overwrite